# <span style="color:darkslateblue;">Text Data Types</span>
There are two ways to store text in pandas:
- <b style="color:red;">object</b> dtype NumPy array
- <b style="color:yellow;">StringDtype</b> extension type

In [1]:
import pandas as pd
import numpy as np

In [2]:
# object dtype remains the default type
pd.Series(["a", "b", "c"])

0    a
1    b
2    c
dtype: object

In [3]:
# to explicitly request string dtype, specify it
pd.Series(["a", "b", "c"], dtype="string")

0    a
1    b
2    c
dtype: string

In [4]:
pd.Series(["a", "b", "c"], dtype=pd.StringDtype())

0    a
1    b
2    c
dtype: string

In [5]:
# or use astype()
pd.Series(["a", "b", "c"]).astype("string")

0    a
1    b
2    c
dtype: string

In [6]:
s = pd.Series(["a", 2, np.nan], dtype="string")
s

0       a
1       2
2    <NA>
dtype: string

In [7]:
type(s[1])

str

In [8]:
# or convert existing data
s1 = pd.Series([1, 2, np.nan], dtype="Int64")
s1

0       1
1       2
2    <NA>
dtype: Int64

In [9]:
s2 = s1.astype("string")
s2

0       1
1       2
2    <NA>
dtype: string

In [10]:
type(s2[0])

str

## Behavior Differences

In [11]:
s = pd.Series(["a", None, "b"], dtype="string")
s

0       a
1    <NA>
2       b
dtype: string

In [12]:
s.str.count("a")

0       1
1    <NA>
2       0
dtype: Int64

In [13]:
s.dropna().str.count("a")

0    1
2    0
dtype: Int64

In [14]:
# when the dtype="object"
s2 = pd.Series(["a", None, "b"], dtype="object")
s2.str.count("a")

0    1.0
1    NaN
2    0.0
dtype: float64

In [15]:
s2.dropna().str.count("a")

0    1
2    0
dtype: int64

In [16]:
s.str.isdigit()

0    False
1     <NA>
2    False
dtype: boolean

In [17]:
s.str.match("a")

0     True
1     <NA>
2    False
dtype: boolean

# <span style="color:darkslateblue;">String Methods</span>
While for a string we can use directly methods such as <b style="color:red;">lower(), upper() or len()</b><br/>
to get the same functionality for a Series or DataFrame we use the <b style="color:yellow;">str</b> accessor<br/>
first and then the method which acts <span style="text-decoration:underline;">element-wise.</span>

In [18]:
s = pd.Series(["A", "B", "C", "Aaba", "Baca", np.nan, "CABA", "dog", "cat"], dtype="string")

In [19]:
s.str.lower()

0       a
1       b
2       c
3    aaba
4    baca
5    <NA>
6    caba
7     dog
8     cat
dtype: string

In [20]:
s.str.upper()

0       A
1       B
2       C
3    AABA
4    BACA
5    <NA>
6    CABA
7     DOG
8     CAT
dtype: string

In [21]:
# NOTE: Since s is a Series (and not a string!), len() returns the
#       length of each element in the Series.
s.str.len()

0       1
1       1
2       1
3       4
4       4
5    <NA>
6       4
7       3
8       3
dtype: Int64

In [22]:
# Same for indexes.
idx = pd.Index([" jack", "jill ", " jesse ", "frank"])

# Strip  white spaces element-wise
idx.str.strip()

Index(['jack', 'jill', 'jesse', 'frank'], dtype='object')

In [23]:
# Strip white spaces from the left side only
idx.str.lstrip()

Index(['jack', 'jill ', 'jesse ', 'frank'], dtype='object')

In [24]:
# or right side
idx.str.rstrip()

Index([' jack', 'jill', ' jesse', 'frank'], dtype='object')

In [25]:
# Remove leading/trailing whitespaces from DataFrames.
df = pd.DataFrame(np.random.randn(3, 2), columns=["    Column A ", " Column B "], index=range(3))
df

,Column A,Column B
0,0.340208,1.602669
1,0.092007,0.119851
2,-0.758855,-1.191071


In [26]:
# Strip white spaces from columns
df.columns.str.strip()

Index(['Column A', 'Column B'], dtype='object')

In [27]:
df.columns.str.lower()

Index(['    column a ', ' column b '], dtype='object')

In [28]:
# Strip leaning/trailing white spaces,
# convert to lower case and
# replace any other whitespace with underscore
# making the changes permanent.
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

df

,column_a,column_b
0,0.340208,1.602669
1,0.092007,0.119851
2,-0.758855,-1.191071


# <span style="color:darkslateblue;">Splitting and Replacing Strings</span>

In [29]:
s2 = pd.Series(["a_b_c", "c_d_e", np.nan, "f_g_h"], dtype="string")

# The split() method acts element-wise on each element of the Series e.g. :
# "a_b_c".split("_") -> ["a", "b", "c"]
# The end result is a Series of lists:
s2.str.split("_")

0    [a, b, c]
1    [c, d, e]
2         <NA>
3    [f, g, h]
dtype: object

In [30]:
# Access elements in the split lists:
s2.str.split("_").str.get(1)

0       b
1       d
2    <NA>
3       g
dtype: object

In [31]:
# Expand split string into separate columns to get a DataFrame
s2.str.split("_", expand=True)

,0,1,2
0,a,b,c
1,c,d,e
2,<NA>,<NA>,<NA>
3,f,g,h


In [32]:
# Limit the number of splits up to 1
s2.str.split("_", expand=True, n=1)

,0,1
0,a,b_c
1,c,d_e
2,<NA>,<NA>
3,f,g_h


In [33]:
# replace() with regex
s3 = pd.Series(
    ["A", "B", "C", "Aaba", "Baca", "", np.nan, "CABA", "dog", "cat"],
    dtype="string",
)

s3

0       A
1       B
2       C
3    Aaba
4    Baca
5        
6    <NA>
7    CABA
8     dog
9     cat
dtype: string

In [34]:
# Match at the start of the string any character followed by a or A
# OR
# match exactly the word dog:
s3.str.replace("^.a|dog", "XX-X ", case=False, regex=True)

0          A
1          B
2          C
3    XX-X ba
4    XX-X ca
5           
6       <NA>
7    XX-X BA
8      XX-X 
9     XX-X t
dtype: string

In [35]:
s4 = pd.Series(["a.b", ".", "b", np.nan, ""], dtype="string")
s4

0     a.b
1       .
2       b
3    <NA>
4        
dtype: string

In [36]:
s4.str.replace(".", "a", regex=True)

0     aaa
1       a
2       a
3    <NA>
4        
dtype: string

In [37]:
dollars = pd.Series(["12", "-$10", "$10,000"], dtype="string")

# The following are equivalent:
dollars.str.replace(r"-\$", "-", regex=True)

0         12
1        -10
2    $10,000
dtype: string

In [38]:
dollars.str.replace("-$", "-", regex=False)

0         12
1        -10
2    $10,000
dtype: string

In [39]:
# Reverse evey lower case word e.g. foo -> oof
pat = r"[a-z]+"


def repl(m):
    return m.group(0)[::-1]


pd.Series(
    ["foo 123", "bar baz", np.nan],
    dtype="string",
).str.replace(pat, repl, regex=True)

0    oof 123
1    rab zab
2       <NA>
dtype: string

In [40]:
# Using regex groups
pat = r"(?P<one>\w+) (?P<two>\w+) (?P<three>\w+)"


def repl(m):
    return m.group("two").swapcase()


pd.Series(["Foo Bar Baz", np.nan], dtype="string").str.replace(pat, repl, regex=True)

0     bAR
1    <NA>
dtype: string

In [41]:
# Use a compiled regular expression object
import re

regex_pat = re.compile(r"^.a|dog", flags=re.IGNORECASE)
s3.str.replace(regex_pat, "XX-XX ", regex=True)

0           A
1           B
2           C
3    XX-XX ba
4    XX-XX ca
5            
6        <NA>
7    XX-XX BA
8      XX-XX 
9     XX-XX t
dtype: string

# <span style="color:darkslateblue;">Concatenation</span>

## Concatenating a Single Series into a String

In [42]:
s = pd.Series(["a", "b", "c", "d"], dtype="string")

s.str.cat(sep=",")

'a,b,c,d'

In [43]:
# Default is sep="" i.e. an empty string
s.str.cat()

'abcd'

In [44]:
t = pd.Series(["a", "b", np.nan, "d"], dtype="string")

# By default, missing values are ignored.
t.str.cat(sep=",")

'a,b,d'

In [45]:
# To give missing values a representation:
t.str.cat(sep=",", na_rep="_")

'a,b,_,d'

## Concatenating a Series and something list-like into a Series

In [46]:
s.str.cat(["A", "B", "C", "D"])

0    aA
1    bB
2    cC
3    dD
dtype: string

In [47]:
s.str.cat(t)

0      aa
1      bb
2    <NA>
3      dd
dtype: string

In [48]:
s.str.cat(t, na_rep="_")

0    aa
1    bb
2    c_
3    dd
dtype: string

## Concatenating a Series and something array-like into a Series

In [49]:
d = pd.concat([t, s], axis=1)
d

,0,1
0,a,a
1,b,b
2,<NA>,c
3,d,d


In [50]:
s.str.cat(d, na_rep="-")

0    aaa
1    bbb
2    c-c
3    ddd
dtype: string

## Concatenating a Series and an Indexed object into a Series, with alignment

In [51]:
u = pd.Series(
    ["b", "d", "a", "c"],
    index=[1, 3, 0, 2],
    dtype="string",
)

In [52]:
s.str.cat(u)

0    aa
1    bb
2    cc
3    dd
dtype: string

In [53]:
v = pd.Series(["z", "a", "b", "d", "e"], index=[-1, 0, 1, 3, 4], dtype="string")

# Use the index from the left side only
s.str.cat(v, join="left", na_rep="-")

0    aa
1    bb
2    c-
3    dd
dtype: string

In [54]:
# Use the union of the two sides
s.str.cat(v, join="outer", na_rep="-")

-1    -z
 0    aa
 1    bb
 2    c-
 3    dd
 4    -e
dtype: string

In [55]:
f = d.loc[[3, 2, 1, 0], :]
print(f)

      0  1
3     d  d
2  <NA>  c
1     b  b
0     a  a


In [56]:
s.str.cat(f, join="left", na_rep="-")

0    aaa
1    bbb
2    c-c
3    ddd
dtype: string

## Concatenating a Series and many objects into a Series

In [57]:
s.str.cat([u, u.to_numpy()], join="left")

0    aab
1    bbd
2    cca
3    ddc
dtype: string

# <span style="color:darkslateblue;">Indexing with .str</span>

In [58]:
s = pd.Series(["A", "B", "C", "Aaba", "Baca", np.nan, "CABA", "dog", "cat"], dtype="string")

In [59]:
s.str[0]

0       A
1       B
2       C
3       A
4       B
5    <NA>
6       C
7       d
8       c
dtype: string

In [60]:
s.str[1]

0    <NA>
1    <NA>
2    <NA>
3       a
4       a
5    <NA>
6       A
7       o
8       a
dtype: string

# <span style="color:darkslateblue;">Extracting substrings</span>

In [63]:
# Extract first match in each subject (extract)
pd.Series(
    ["a1", "b2", "c3"],
    dtype="string",
).str.extract(
    r"([ab])(\d)",
    expand=False,
)

,0,1
0,a,1
1,b,2
2,<NA>,<NA>


In [66]:
# Extracting named groups
pd.Series(["a1", "b2", "c3"], dtype="string").str.extract(
    r"(?P<letter>[ab])(?P<digit>\d)",
    expand=False,
)

,letter,digit
0,a,1
1,b,2
2,<NA>,<NA>


In [68]:
# Optional groups
pd.Series(["a1", "b2", "3"], dtype="string").str.extract(
    r"([ab])?(\d)",
    expand=False,
)

,0,1
0,a,1
1,b,2
2,<NA>,3


In [70]:
pd.Series(["a1", "b2", "c3"], dtype="string").str.extract(
    r"[ab](\d)",
    expand=True,
)

,0
0,1
1,2
2,<NA>


In [73]:
s = pd.Series(["a1", "b2", "c3"], ["A11", "B22", "C33"], dtype="string")
print(s)
print(s.index)

A11    a1
B22    b2
C33    c3
dtype: string
Index(['A11', 'B22', 'C33'], dtype='object')


In [ ]:
# If the regex has exactly one capture group AND expand=True -> DataFrame with 1 column
s.index.str.extract("(?P<letter>[a-zA-Z])", expand=True)

,letter
0,A
1,B
2,C


In [ ]:
# If the regex has exactly one capture group AND expand=False -> an Index
s.index.str.extract("(?P<letter>[a-zA-Z])", expand=False)

Index(['A', 'B', 'C'], dtype='object', name='letter')

In [77]:
# If the regex has many capture groups AND expand=True -> DataFrame
s.index.str.extract(r"(?P<letter>[a-zA-Z])(\d+)", expand=True)

,letter,1
0,A,11
1,B,22
2,C,33


In [ ]:
# Extract all matches in each subject (extractall)
s = pd.Series(
    ["a1a2", "b1", "c1"],
    index=["A", "B", "C"],
    dtype="string",
)

# a regex pattern with two capturing groups
two_groups = "(?P<letter>[a-z])(?P<digit>[0-9])"

# extract() returns only the 1st match
s.str.extract(two_groups, expand=True)

,letter,digit
A,a,1
B,b,1
C,c,1


In [80]:
# extractall() returns all matches
s.str.extractall(two_groups)

letter digit
  match             
A 0          a     1
  1          a     2
B 0          b     1
C 0          c     1

# <span style="color:darkslateblue;">Testing for strings that match or contain a pattern</span>

In [82]:
pattern = r"[0-9][a-z]"

pd.Series(
    ["1", "2", "3a", "3b", "03c", "4dx"],
    dtype="string",
).str.contains(pattern)

0    False
1    False
2     True
3     True
4     True
5     True
dtype: boolean

In [83]:
pd.Series(
    ["1", "2", "3a", "3b", "03c", "4dx"],
    dtype="string",
).str.match(pattern)

0    False
1    False
2     True
3     True
4    False
5     True
dtype: boolean

In [84]:
pd.Series(
    ["1", "2", "3a", "3b", "03c", "4dx"],
    dtype="string",
).str.fullmatch(pattern)

0    False
1    False
2     True
3     True
4    False
5    False
dtype: boolean